# PRC Eurocontrol ML Pipeline, adapted for Harsh's enrichment handover (2026-10-07)

This notebook was rewritten to use `enriched_features_train.parquet` / `enriched_features_ranking.parquet`
(see `HANDOVER.md`) instead of hand-rolled features computed from `cleaned_full_data.parquet`.

**What changed vs the previous version of this notebook:**
- The manual time features (`hour`, `dayofweek`, `delay_min`), traffic-congestion counts
  (`push_prev_*m`, `arr_prev_*m`, `rwy_push_prev_*m`) and the ad-hoc "airport+runway median taxi"
  baseline are gone. The enrichment file now provides fixed, leakage-safe versions of all of these
  (`pushback_hour`, `dep_delay_min`, `planes_pushed_last_*min`, `arrivals_last_30min`,
  `typical_taxi_sec`, `unimpeded_taxi_sec`), plus weather and stand-to-runway distance, which we
  didn't have before.
- `STAND_mvt` and `WK_TBL_CAT_flt` are replaced by the enrichment's `stand_area` and
  `aircraft_size`, which group rare values instead of leaving the model to overfit them.
- **The train/valid/test split was fixed.** The old notebook trained on Jan through Oct and tested
  on Nov + Dec, i.e. it just took the first months of the year for training and the last ones for
  testing. That split made sense for a generic time series, but not for this problem: all of the
  2025 training data already sits before the 2026 ranking period, so a strict chronological cutoff
  inside 2025 buys no protection against leakage, and it quietly built a validation set (Nov, Dec)
  that doesn't look like the thing we're actually scored on (January and July). Aviation traffic is
  strongly seasonal, so "last two months of the year" and "January and July" are different regimes.
  The split below instead holds out January and July of 2025, the same months as the ranking set,
  for the final test evaluation, matching the approach the handover itself used for its own quick
  model test (section 4: "train on other months, test on Jan + Jul").
- `month` is left out of the feature list until after the holdout evaluation, then added back for
  the model trained on all of 2025 for the actual 2026 predictions.

**Decided against:** take-off time minus `AOBT_3_flt` (handover section 6.1). Tracing the columns
shows `TAXITIME_SEC_mvt` (the target) equals `MVT_TIME_UTC_mvt` − `BLOCK_TIME_UTC_mvt` exactly;
`BLOCK_TIME_UTC_mvt` is deliberately blanked out in the ranking file, but `MVT_TIME_UTC_mvt`
(the actual take-off time) is not. So this feature is a noisy reconstruction of the answer
(r = 0.64 against the true target), not a genuine predictor — it's why it drops Harsh's test RMSE
so much (315→276), and why it's excluded from the final model rather than added. See
`FINAL_PHASE_HANDOVER.md` section 5 for the full trace.

**Running locally, not in Colab**, since there's no need for Drive at all: the raw monthly files
and the enrichment outputs can just sit next to this notebook, the same "same folder" convention
`enrichment.py` already documents for its own inputs. The path setup below works on any machine
(Windows, Mac, Linux) and for any teammate, as long as the files are in that one folder, or you
point `PRC_DATA_DIR` at wherever they are, with no hardcoded usernames or drive letters.
There's also no local GPU here, so the model trains on CPU (`device='cpu'`), which will be slower
than the GPU runs the original notebook's numbers came from; switch it back to `'cuda'` if you run
this somewhere with a GPU.


In [ ]:
import os
from pathlib import Path

# Everything (the 12 training_2025-*.parquet files, ranking.parquet, and the enrichment
# outputs) is expected in one folder, next to this notebook. Override with the PRC_DATA_DIR
# environment variable if you keep them somewhere else, no code edit needed.
BASE_DIR = Path(os.environ.get('PRC_DATA_DIR', Path.cwd())).resolve()
RAW_DIR = BASE_DIR
DATA_DIR = BASE_DIR

assert BASE_DIR.is_dir(), f'{BASE_DIR} does not exist, set the PRC_DATA_DIR environment variable to the right folder'
print('BASE_DIR:', BASE_DIR)
print('contents:', sorted(p.name for p in BASE_DIR.iterdir()))

RANKING_FILE = RAW_DIR / 'ranking.parquet'
HAVE_RANKING = RANKING_FILE.exists()
if not HAVE_RANKING:
    print(f"ranking.parquet not found at {RANKING_FILE}: ranking-set cells further down will be skipped")


In [ ]:
import glob
import pandas as pd

# raw columns we still need straight from the challenge files: the target, and the
# categoricals the handover recommends giving the model on top of the 19 enriched features
RAW_TRAIN_COLS = ['MVT_ID_mvt', 'PHASE_mvt', 'ADEP_mvt', 'RUNWAY_mvt',
                   'AIRCRAFT_TYPE_mvt', 'AIRCRAFT_OPERATOR_flt', 'TAXITIME_SEC_mvt']
RAW_RANK_COLS = [c for c in RAW_TRAIN_COLS if c != 'TAXITIME_SEC_mvt']

def load_raw(files, cols):
    parts = []
    for f in files:
        df = pd.read_parquet(f)
        df = df[df['PHASE_mvt'] == 'DEP']
        parts.append(df[[c for c in cols if c in df.columns]])
    return pd.concat(parts, ignore_index=True)

train_raw = load_raw(sorted(glob.glob(os.path.join(RAW_DIR, 'training_2025-*.parquet'))), RAW_TRAIN_COLS)
enr_train = pd.read_parquet(os.path.join(DATA_DIR, 'enriched_features_train.parquet'))
dep = train_raw.merge(enr_train, on='MVT_ID_mvt', how='inner')
print('train:', dep.shape)
assert len(dep) == len(train_raw) == len(enr_train), 'train merge dropped rows, check MVT_ID_mvt join'

if HAVE_RANKING:
    rank_raw = load_raw([RANKING_FILE], RAW_RANK_COLS)
    enr_rank = pd.read_parquet(os.path.join(DATA_DIR, 'enriched_features_ranking.parquet'))
    rank = rank_raw.merge(enr_rank, on='MVT_ID_mvt', how='inner')
    print('ranking:', rank.shape)
    assert len(rank) == len(rank_raw) == len(enr_rank), 'ranking merge dropped rows, check MVT_ID_mvt join'
else:
    rank = None


Sanity check against the handover: train should be 2,085,047 rows (and ranking, once you have
the file, 344,841 rows). The `assert`s above confirm every raw DEP row found its enrichment row
(and vice versa) via `MVT_ID_mvt`. If either fires, the raw files and the enrichment files are out
of sync (re-run `enrichment.py` against the current raw files before continuing).


In [ ]:
# same filter as before: 1-second and 12-hour taxi times can't be real.
# only applied to the training target: the ranking set has no TAXITIME_SEC_mvt to filter on.
dep = dep[dep['TAXITIME_SEC_mvt'].between(60, 7200)].copy()
print(len(dep))


Notes (unchanged from the original pipeline):

2.08M departures, median taxi-out about 912s (15 min), mean 990s.

1-second and 42,661-second (12h) taxi times can't happen, kept only 60 to 7200s.


In [ ]:
import numpy as np

frames = [dep] + ([rank] if rank is not None else [])

# harmonise dtypes: the handover's output has is_fog/is_snow_ice as float64 in train but
# int32 in ranking (same values, different dtype). cast both to float64 so the model sees
# one consistent dtype.
for d in frames:
    d['is_fog'] = d['is_fog'].astype('float64')
    d['is_snow_ice'] = d['is_snow_ice'].astype('float64')

# make runway (and therefore its congestion signal) airport-specific, same trick as before:
# it was a cheap win in the old pipeline (301.7 vs 302.2s) and costs nothing to keep.
for d in frames:
    d['RUNWAY_mvt'] = (d['ADEP_mvt'].astype(str) + '_' + d['RUNWAY_mvt'].astype(str))

# raw categoricals (per handover section 2) plus the enrichment's own grouped categoricals.
# STAND_mvt and WK_TBL_CAT_flt are deliberately left out in favour of stand_area / aircraft_size.
cat_features = ['ADEP_mvt', 'RUNWAY_mvt', 'AIRCRAFT_TYPE_mvt', 'AIRCRAFT_OPERATOR_flt',
                'stand_area', 'aircraft_size']

# the 19 enriched numeric features minus the 2 that became categoricals above, and minus
# month (held back deliberately, see the split below)
num_features = ['pushback_hour', 'is_weekend',
                'planes_pushed_last_15min', 'planes_pushed_last_30min', 'planes_pushed_last_60min',
                'arrivals_last_30min', 'dep_delay_min', 'aobt_missing',
                'visibility_miles', 'wind_kt', 'temp_c', 'is_fog', 'is_snow_ice',
                'typical_taxi_sec', 'unimpeded_taxi_sec', 'stand_runway_km']

for c in cat_features:
    dep[c] = dep[c].astype('category')
if rank is not None:
    for c in cat_features:
        rank[c] = rank[c].astype('category')
        # ranking categories must be a subset of train's (handover confirms this holds), align
        # them explicitly so XGBoost sees the same category codes on both sides
        rank[c] = rank[c].cat.set_categories(dep[c].cat.categories)

X = dep[cat_features + num_features]
y = dep['TAXITIME_SEC_mvt']
X_rank = rank[cat_features + num_features] if rank is not None else None
print(X.shape)
print(y.describe())


In [ ]:
from sklearn.model_selection import train_test_split

# Test = January and July 2025, the same two months as the 2026 ranking set. This is the
# fix for the split issue: holding out the calendar's last two months (Nov, Dec) tested the
# model on a seasonal regime (winter holidays) that has nothing to do with what it will
# actually be scored on (Jan + Jul). Holding out Jan + Jul instead means the test score is a
# real estimate of ranking performance, not just "does it generalise to late autumn".
test_mask = dep['month'].isin([1, 7])

# the other 10 months aren't chronologically ordered relative to the test months any more
# (Jan and Jul sit in the middle of the year, not at an edge), so a plain random split is
# fine for the validation set used for early stopping; there's no extra leakage risk versus
# a chronological split, since every bit of 2025 predates the 2026 ranking set either way.
train_idx, valid_idx = train_test_split(dep.index[~test_mask], test_size=0.1, random_state=42)

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_valid, y_valid = X.loc[valid_idx], y.loc[valid_idx]
X_test,  y_test  = X[test_mask],     y[test_mask]
print(len(X_train), len(X_valid), len(X_test))


In [ ]:
import xgboost as xgb

def rmse(a, b):
    return np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

# median taxi-out per airport+runway, learned on train only, same baseline as before, for comparison
key_train = X_train['ADEP_mvt'].astype(str) + '_' + X_train['RUNWAY_mvt'].astype(str)
key_valid = X_valid['ADEP_mvt'].astype(str) + '_' + X_valid['RUNWAY_mvt'].astype(str)
medians = y_train.groupby(key_train).median()
baseline_pred = key_valid.map(medians).fillna(y_train.median())
print('Baseline RMSE (valid):', round(rmse(y_valid, baseline_pred), 1))

model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    enable_categorical=True,
    max_cat_to_onehot=1,
    eval_metric='rmse',
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1,
    device='cpu',  # no GPU on this machine; switch to 'cuda' if you run this somewhere with one
)

model.fit(X_train, y_train,
          eval_set=[(X_valid, y_valid)],
          verbose=200)

print('Best iteration:', model.best_iteration)
print('XGBoost RMSE (valid):', round(rmse(y_valid, model.predict(X_valid)), 1))


Notes: *(fill in after you run this cell, I generated this notebook without running it since
I don't have a GPU here and training 2M rows on CPU takes a while. The old pipeline's final RMSE
was 299.1s valid / 308.4s test on a Nov/Dec holdout with 18 hand-built features, not directly
comparable any more since the holdout months changed. Harsh's own quick LightGBM test on the 19
enriched features plus raw airport/runway/aircraft-type, also tested on Jan + Jul, got 315.4s,
which is the more relevant number to compare against here.)*


In [ ]:
# test on January + July, same diagnostics as before: overall RMSE, baseline for comparison,
# per-airport breakdown, feature importance
test_pred = model.predict(X_test)
print('XGBoost RMSE (test):', round(rmse(y_test, test_pred), 1))

key_test = X_test['ADEP_mvt'].astype(str) + '_' + X_test['RUNWAY_mvt'].astype(str)
baseline_test = key_test.map(medians).fillna(y_train.median())
print('Baseline RMSE (test):', round(rmse(y_test, baseline_test), 1))

per_airport = (pd.DataFrame({'airport': X_test['ADEP_mvt'], 'err2': (y_test - test_pred) ** 2})
               .groupby('airport', observed=True)['err2'].mean() ** 0.5)
print(per_airport.round(1).sort_values())

imp = pd.Series(model.get_booster().get_score(importance_type='gain'))
print((imp / imp.sum()).sort_values(ascending=False).round(3))


In [ ]:
# handover section 6.2: January 2026 has about 3x the snow of January 2025 (4.6% vs 1.6% of
# departures). The test set above now includes January, so this check is directly relevant:
# if snowy flights carry a disproportionate share of the error, expect the 2026 ranking score
# to be worse than this test RMSE suggests, purely because 2026 has more snow.
snow_mask = X_test['is_snow_ice'] > 0
print('snowy flights in Jan+Jul test:', snow_mask.sum(), f'({snow_mask.mean():.1%})')
print('RMSE snow:', round(rmse(y_test[snow_mask], test_pred[snow_mask]), 1))
print('RMSE no snow:', round(rmse(y_test[~snow_mask], test_pred[~snow_mask]), 1))


If snowy-flight RMSE is much worse than clean-weather RMSE, expect the January 2026 ranking
score to be pulled down by the higher snow share, worth flagging in the writeup either way.


In [ ]:
import matplotlib.pyplot as plt

valid_pred = model.predict(X_valid)
resid = y_valid - valid_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
hb = ax.hexbin(y_valid / 60, valid_pred / 60, gridsize=80, bins='log', cmap='viridis',
               extent=(0, 50, 0, 50))
ax.plot([0, 50], [0, 50], 'r--', lw=1)
ax.set(xlabel='Actual taxi-out (min)', ylabel='Predicted (min)', title='Predicted vs actual')
fig.colorbar(hb, ax=ax, label='log(count)')

ax = axes[0, 1]
ax.hist(resid / 60, bins=100, range=(-20, 30), color='steelblue')
ax.axvline(0, color='r', ls='--', lw=1)
ax.set(xlabel='Actual minus predicted (min)', ylabel='Flights', title='Error distribution')

ax = axes[1, 0]
per_ap = pd.DataFrame({
    'airport': X_valid['ADEP_mvt'].astype(str),
    'xgb': (y_valid - valid_pred) ** 2,
    'baseline': (y_valid - baseline_pred) ** 2,
}).groupby('airport')[['baseline', 'xgb']].mean() ** 0.5
per_ap.sort_values('xgb').plot.barh(ax=ax, color=['lightgray', 'steelblue'])
ax.set(xlabel='RMSE (s)', ylabel='', title='RMSE per airport')

ax = axes[1, 1]
(imp / imp.sum()).sort_values().plot.barh(ax=ax, color='steelblue')
ax.set(xlabel='Share of total gain', title='Feature importance')

plt.tight_layout()
plt.show()


Tuning the model next

In [ ]:
!pip install -q optuna
import optuna

Xs = X_train.sample(400_000, random_state=42)
ys = y_train.loc[Xs.index]

def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 100, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 50, log=True),
    }
    m = xgb.XGBRegressor(
        n_estimators=2000, learning_rate=0.1,
        tree_method='hist', enable_categorical=True, max_cat_to_onehot=1,
        eval_metric='rmse', early_stopping_rounds=50,
        random_state=42, n_jobs=-1, **params)
    m.fit(Xs, ys, eval_set=[(X_valid, y_valid)], verbose=False)
    return rmse(y_valid, m.predict(X_valid))

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.enqueue_trial({'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.8,
                     'colsample_bytree': 0.8, 'reg_lambda': 1})
study.optimize(objective, n_trials=10, show_progress_bar=True)

print('Current settings RMSE:', round(study.trials[0].value, 1))
print('Best RMSE:', round(study.best_value, 1))
print('Best settings:', study.best_params)


In [ ]:
best = study.best_params

model_tuned = xgb.XGBRegressor(
    n_estimators=3000, learning_rate=0.05,
    tree_method='hist', enable_categorical=True, max_cat_to_onehot=1,
    eval_metric='rmse', early_stopping_rounds=100,
    random_state=42, n_jobs=-1, **best)

model_tuned.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=200)

print('Best iteration:', model_tuned.best_iteration)
print('Tuned RMSE (valid):', round(rmse(y_valid, model_tuned.predict(X_valid)), 1))


In [ ]:
import joblib

params = study.best_params
# add `month` back in now that the holdout evaluation above is done
final_num_features = num_features + ['month']
X_full = dep[cat_features + final_num_features]

n_trees = int(model_tuned.best_iteration * 1.1)
final_model = xgb.XGBRegressor(
    n_estimators=n_trees, learning_rate=0.05,
    tree_method='hist', enable_categorical=True, max_cat_to_onehot=1,
    random_state=42, n_jobs=-1, device='cpu', **params)
final_model.fit(X_full, y)

joblib.dump({
    'model': final_model,
    'cat_features': cat_features,
    'num_features': final_num_features,
    'categories': {c: list(X_full[c].cat.categories) for c in cat_features},
    'params': params,
    'n_trees': n_trees,
}, os.path.join(DATA_DIR, 'taxi_out_xgb.joblib'))

# joblib/pickle ties the saved file to the exact xgboost version that wrote it (loading it on a
# different version prints a warning and can break). also save the booster itself in xgboost's
# own portable format, which is safe to move across machines and xgboost versions.
final_model.save_model(os.path.join(DATA_DIR, 'taxi_out_xgb.ubj'))
print('saved, trees:', n_trees)


In [ ]:
# predictions for the final-phase ranking set (Jan/Feb/Jun/Jul 2026).
if rank is not None:
    X_rank_full = rank[cat_features + final_num_features]
    rank_pred = final_model.predict(X_rank_full)
    submission = pd.DataFrame({
        'MVT_ID_mvt': rank['MVT_ID_mvt'],
        'predicted_taxi_sec': rank_pred,
    })
    submission.to_parquet(os.path.join(DATA_DIR, 'ranking_predictions.parquet'), index=False)
    print(submission.shape)
else:
    print('ranking.parquet was not found earlier, nothing to predict on yet. '
          'get that file, re-run the loading cell near the top, then re-run this cell.')


In [ ]:
# reshape to the organizers' exact submission schema/row order (final_submitting.parquet,
# copied into this folder as submitting.parquet): MVT_ID_mvt + TAXITIME_SEC_mvt, every row
# filled, same order as the template.
if rank is not None:
    sub_template = pd.read_parquet(os.path.join(DATA_DIR, 'submitting.parquet'))
    final_submission = sub_template[['MVT_ID_mvt']].merge(
        submission.rename(columns={'predicted_taxi_sec': 'TAXITIME_SEC_mvt'}),
        on='MVT_ID_mvt', how='left')
    final_submission['TAXITIME_SEC_mvt'] = final_submission['TAXITIME_SEC_mvt'].astype('float64')

    assert len(final_submission) == len(sub_template)
    assert list(final_submission['MVT_ID_mvt']) == list(sub_template['MVT_ID_mvt'])
    assert final_submission['TAXITIME_SEC_mvt'].notna().all()
    assert (final_submission['TAXITIME_SEC_mvt'] > 0).all()

    final_submission.to_parquet(os.path.join(DATA_DIR, 'submission_final_phase.parquet'), index=False)
    print('saved submission_final_phase.parquet', final_submission.shape)


## Notes (handover section 6)

1. **Take-off time minus `AOBT_3_flt`** — decided against, not an open question anymore.
   `TAXITIME_SEC_mvt` equals `MVT_TIME_UTC_mvt` − `BLOCK_TIME_UTC_mvt` exactly in training data
   (checked). `BLOCK_TIME_UTC_mvt` is deliberately blanked in the ranking file; `MVT_TIME_UTC_mvt`
   (actual take-off time) is not, even though it marks the *end* of the period being predicted.
   `AOBT_3_flt` only correlates with the true `BLOCK_TIME_UTC_mvt` (r = 0.64 against the target),
   so this "feature" is a noisy answer-key leak, not a real predictor — excluded on that basis,
   not pending rules confirmation. See `FINAL_PHASE_HANDOVER.md` section 5.
2. **January 2026 snow** is about 3x January 2025's, see the snow-flight RMSE check above. Checked
   the other three final-phase months too (`FINAL_PHASE_HANDOVER.md` section 4): only January has
   an adverse weather shift vs. training; Feb/Jun/Jul are flat or calmer than their 2025 counterparts.
